#### En estos ejercicios vemos algunos algoridmos y formas de usarlos para encontrar cedulas dentro de un archivo csv el cual tiene millones de filas en cada fase re realiza un proceso de mejora el cual optimiza mucho el tiempo de rendimiento para encontrar lo que nesecitamos al igual que mejora tambien la redaccion y funcionamiento del codigo como esta en el ejemplo del drive en este ejemplo con pocos datos sse mostrara solo el proceso de como se hace o como se encuentran pero sin ser tan extrictamente complejo para ello te recomiendo descargar los datos para ver resultados mas reales o mass calculados para ver las diferencias de rendimiento en cada proceso

In [1]:
import polars as pl
archivo = pl.read_csv("cedulas-simuladas-total1.csv")
print(archivo.dtypes)

[String, Int64, Int64, String, String]


Este programa carga un archivo CSV con cédulas usando Polars y luego onvierte esa columna en una lista para ordenarla manualmente, lo cual puede ser un proceso algo lento dependiendo del tamaño de los datos. Después de ordenar, se aplica una búsqueda binaria para encontrar una cédula específica de forma más eficiente y, si existe, se muestran este algoridmo es medianamente rapido gracias a que hace particiones por la mitad del dataset y por maximo se demora ma o menos 20 iteraicones hasta encontrar cualquier cedula como se muestra al ejecutar el codigo


In [2]:
%%time
import polars as pl
# Cargo el archivo CSV y aseguro que la columna 'cedula' sea tipo texto
archivo = pl.read_csv("cedulas-simuladas-total1.csv", dtypes={"cedula": pl.Utf8})

# Extraigo la columna de cédulas y la convierto en una lista de Python
cedulas = archivo["cedula"].to_list()

# Ordeno la lista manualmente para poder aplicar búsqueda binaria
# Este paso puede ser algo costoso si el archivo es muy grande
cedulas.sort()


# Defino la función de búsqueda binaria
def busqueda_binaria(lista, objetivo):
    # Inicializo los índices izquierdo y derecho
    izq, der = 0, len(lista) - 1
    pasos = 0

    # Mientras el rango sea válido
    while izq <= der:
        pasos += 1

        # Calculo el índice medio
        medio = (izq + der) // 2
        valor_medio = lista[medio]

        # Muestro el estado actual (útil para entender el proceso)
        print(f"Paso {pasos}: izquierda={izq}, derecha={der}, medio={medio}, valor_medio={valor_medio}")

        # Si encuentro el valor, retorno True
        if valor_medio == objetivo:
            return True

        # Si el valor medio es menor, busco en la mitad derecha
        elif valor_medio < objetivo:
            izq = medio + 1

        # Si es mayor, busco en la mitad izquierda
        else:
            der = medio - 1

    # Si no se encuentra, retorno False
    return False


# Punto de entrada del programa
if __name__ == "__main__":
    print("Archivo cargado correctamente.")
    print(f"Total de cédulas cargadas: {len(cedulas):,}")
    objetivo = input("Ingresa la cédula que deseas buscar: ")
    encontrado = busqueda_binaria(cedulas, objetivo)

    # Si la cédula existe
    if encontrado:
        print(f"La cédula {objetivo} SÍ está en el archivo.")
        resultado = archivo[archivo["cedula"] == objetivo]
        print(f"datos de la persona: {resultado}")
    else:
        # Si no existe, informo al usuario
        print(f"La cédula {objetivo} NO está en el archivo.")



Archivo cargado correctamente.
Total de cédulas cargadas: 10,000


<timed exec>:3: DeprecationWarning: the argument `dtypes` for `read_csv` is deprecated. It was renamed to `schema_overrides` in version 0.20.31.


Paso 1: izquierda=0, derecha=9999, medio=4999, valor_medio=25489892
Paso 2: izquierda=0, derecha=4998, medio=2499, valor_medio=1352077712
Paso 3: izquierda=0, derecha=2498, medio=1249, valor_medio=1172290479
Paso 4: izquierda=1250, derecha=2498, medio=1874, valor_medio=1261961820
Paso 5: izquierda=1250, derecha=1873, medio=1561, valor_medio=1216086012
Paso 6: izquierda=1562, derecha=1873, medio=1717, valor_medio=1238845034
Paso 7: izquierda=1562, derecha=1716, medio=1639, valor_medio=1228058634
Paso 8: izquierda=1640, derecha=1716, medio=1678, valor_medio=1233448013
Paso 9: izquierda=1640, derecha=1677, medio=1658, valor_medio=1230608862
Paso 10: izquierda=1659, derecha=1677, medio=1668, valor_medio=1231467659
Paso 11: izquierda=1659, derecha=1667, medio=1663, valor_medio=1230982
Paso 12: izquierda=1664, derecha=1667, medio=1665, valor_medio=1231336425
Paso 13: izquierda=1664, derecha=1664, medio=1664, valor_medio=1231142217
La cédula 12312311 NO está en el archivo.
CPU times: user 48.

en este proceso lo realizamo con la misma bisqueda binaria pero utilizando bisect y otras herramientas para determinar la cedula haciendo iteraicones de una forma muy parecida al primer codigo

In [3]:
%%time
import polars as pl
import bisect

archivo = pl.read_csv("cedulas-simuladas-total1.csv")
cedulas = archivo["cedula"].cast(pl.Utf8).to_list()

# Ordeno la lista, paso necesario para usar búsqueda binaria
# Este proceso puede tardar dependiendo del tamaño del dataset
cedulas.sort()

# Función de búsqueda binaria usando el módulo bisect
def busqueda_binaria(lista, objetivo):
    # Encuentra la posición donde debería estar el valor
    indice = bisect.bisect_left(lista, objetivo)

    # Verifica si el valor realmente está en esa posición
    return indice < len(lista) and lista[indice] == objetivo


# Punto de entrada del programa
if __name__ == "__main__":
    print(f"Total de cédulas cargadas: {len(cedulas):,}\n")
    objetivo = input("Ingresa la cédula que deseas buscar: ").strip()
    encontrado = busqueda_binaria(cedulas, objetivo)
    if encontrado:
        print(f"La cédula {objetivo} está en el archivo.\n")
        resultado = archivo.filter(pl.col("cedula").cast(pl.Utf8) == objetivo)
        print(resultado)
    else:
        # Si no se encuentra
        print(f"La cédula {objetivo} no está en el archivo.")




Total de cédulas cargadas: 10,000

La cédula 12313212231 no está en el archivo.
CPU times: user 38.3 ms, sys: 6.37 ms, total: 44.7 ms
Wall time: 2.36 s


En este caso se utiliza directamente polars con su funcion de filter para encontrar de manera eficiente la cedula ya que polars tras de usar mas nuestra ram puede hacer mas procesos de busqueda para ser mas eficiente y esto se denota en el tiempo de ejecucion 

In [12]:
%%time
import polars as pl

archivo = pl.read_csv("cedulas-simuladas-total1.csv")

if __name__ == "__main__":
    print(f"Total de cédulas cargadas: {len(archivo):,}\n")

    # Solicito al usuario la cédula a buscar
    objetivo = input("Ingresa la cédula que deseas buscar: ").strip()

    # Convierto la columna 'cedula' a texto y luego a lista de Python
    # Este paso puede consumir memoria si el dataset es grande
    lista = archivo["cedula"].cast(pl.Utf8).to_list()

    # Verifico si la cédula está en la lista usando el operador 'in'
    # Esta búsqueda es lineal (O(n)), por lo que puede ser lenta con muchos datos
    encontrado = objetivo in lista

    # Si la cédula existe
    if encontrado:
        print(f"La cédula {objetivo} está en el archivo.\n")

        # Filtro el DataFrame para obtener todos los datos asociados
        resultado = archivo.filter(pl.col("cedula").cast(pl.Utf8) == objetivo)
        print(resultado)
    else:
        # Si no existe, informo al usuario
        print(f"La cédula {objetivo} no está en el archivo.")




Total de cédulas cargadas: 4,000,001

La cédula 2571945 está en el archivo.

shape: (1, 5)
┌───────────────┬──────┬─────────┬──────┬──────────────────────┐
│ nombre        ┆ edad ┆ cedula  ┆ sexo ┆ correo               │
│ ---           ┆ ---  ┆ ---     ┆ ---  ┆ ---                  │
│ str           ┆ i64  ┆ i64     ┆ str  ┆ str                  │
╞═══════════════╪══════╪═════════╪══════╪══════════════════════╡
│ Andrés García ┆ 72   ┆ 2571945 ┆ M    ┆ usuario2@example.com │
└───────────────┴──────┴─────────┴──────┴──────────────────────┘
CPU times: user 1.32 s, sys: 329 ms, total: 1.65 s
Wall time: 2.29 s


En este caso ya no volvemos a crear una lista de las cedulas sino que directamente utilizamos el archivo csv para no hacer un doble proceso y de esta forma optimizamos el tiempo aun mas utilizando completamente polars en el proceso a veces no hay nesecidad de hacer ese cambio de dato numerico a texto de la cedula ya que polars lo hace de otra forma 

In [7]:
%%time
import polars as pl

archivo = pl.read_csv("cedulas-simuladas-total1.csv")

if __name__ == "__main__":
    print(f"Total de cédulas cargadas: {len(archivo):,}\n")

    # Solicito la cédula a buscar
    objetivo = input("Ingresa la cédula que deseas buscar: ").strip()

    # Filtro directamente sobre el DataFrame sin convertir a lista
    # Esto evita consumir memoria extra y aprovecha la eficiencia de Polars
    objetivo = int(objetivo)
    resultado = archivo.filter(pl.col("cedula") == objetivo)

    # Si se encuentran coincidencias
    if resultado.height > 0:
        print(f"La cédula {objetivo} está en el archivo.")
        print(f"Total de coincidencias: {resultado.height}\n")
        print(resultado)
    else:
        # Si no hay resultados
        print(f"La cédula {objetivo} no está en el archivo.")


Total de cédulas cargadas: 10,000

La cédula 7350753 está en el archivo.
Total de coincidencias: 1

shape: (1, 5)
┌────────────┬──────┬─────────┬──────┬──────────────────────┐
│ nombre     ┆ edad ┆ cedula  ┆ sexo ┆ correo               │
│ ---        ┆ ---  ┆ ---     ┆ ---  ┆ ---                  │
│ str        ┆ i64  ┆ i64     ┆ str  ┆ str                  │
╞════════════╪══════╪═════════╪══════╪══════════════════════╡
│ Luis Silva ┆ 86   ┆ 7350753 ┆ M    ┆ usuario7@example.com │
└────────────┴──────┴─────────┴──────┴──────────────────────┘
CPU times: user 16.3 ms, sys: 6.02 ms, total: 22.3 ms
Wall time: 1.16 s


Este es otro ejemplo usando completamente polars pero esta vez estamos buscando por edades denotando que no solo se puede buscar un unico dato con polars sino muchos que tengan la misma relacion

In [5]:
%%time
import polars as pl

archivo = pl.read_csv("cedulas-simuladas-total1.csv")


if __name__ == "__main__":
    
    # Muestro la cantidad total de registros
    print(f"Total de cédulas cargadas: {len(archivo):,}\n")

    # Solicito al usuario una edad a buscar
    objetivo = input("Ingresa la edad que deseas buscar: ").strip()

    # Filtro el DataFrame por la columna 'edad'
    # Aquí convierto a texto para comparar con el input del usuario
    objetivo = int(objetivo)
    resultado = archivo.filter(pl.col("edad") == objetivo)

    # Si se encuentran coincidencias
    if resultado.height > 0:
        print(f"Se encontraron {resultado.height} registros con edad {objetivo}:\n")

        # Muestro los resultados encontrados
        print(resultado)
    else:
        # Si no hay coincidencias
        print(f"No se encontraron registros con edad {objetivo}.")



Total de cédulas cargadas: 10,000

Se encontraron 142 registros con edad 40:

shape: (142, 5)
┌───────────────────┬──────┬────────────┬──────┬─────────────────────────┐
│ nombre            ┆ edad ┆ cedula     ┆ sexo ┆ correo                  │
│ ---               ┆ ---  ┆ ---        ┆ ---  ┆ ---                     │
│ str               ┆ i64  ┆ i64        ┆ str  ┆ str                     │
╞═══════════════════╪══════╪════════════╪══════╪═════════════════════════╡
│ Valentina Ramírez ┆ 40   ┆ 1556152900 ┆ F    ┆ usuario81@example.com   │
│ Miguel Rojas      ┆ 40   ┆ 81690398   ┆ M    ┆ usuario121@example.com  │
│ Andrés Ramírez    ┆ 40   ┆ 31009452   ┆ M    ┆ usuario130@example.com  │
│ Sofía Pérez       ┆ 40   ┆ 1546272321 ┆ M    ┆ usuario291@example.com  │
│ Andrés Ortiz      ┆ 40   ┆ 1039019611 ┆ F    ┆ usuario342@example.com  │
│ …                 ┆ …    ┆ …          ┆ …    ┆ …                       │
│ Paula Martínez    ┆ 40   ┆ 4194969    ┆ M    ┆ usuario9709@example.com │
│ Luis